# Topic: SQL DENSE_RANK() Pattern

## Definition (30-second explanation)
* `DENSE_RANK()` is a window function that assigns a rank to each row within a partition, but unlike `RANK()`, it **never skips rank numbers** when there are ties.
* If two rows tie for rank 2, the next row immediately gets rank 3. The ranks are "packed densely" without gaps.

## Why Interviewers Ask This
* It is the definitive solution to the classic "Find the Nth highest [value]" interview question.
* To test if you understand the nuanced, practical differences between `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`.
* To see if you can handle tie-breaker scenarios without accidentally dropping valid tied rows (which happens if you use `ROW_NUMBER`).

## Core Concepts
* **No Gaps:** Ties share the exact same rank, and the sequence continues with the very next integer.
* **PARTITION BY:** Divides the result set into groups. The rank resets to 1 at the start of each partition.
* **CTE/Subquery Requirement:** Like all window functions, you must evaluate it inside a CTE or subquery before you can filter on the rank (e.g., `WHERE dense_rank_col = N`).

## When to Use
* Finding the Nth highest/lowest value where ties must be included (e.g., all employees who share the 2nd highest salary).
* Leaderboard and competition scenarios where tied scores should share the same rank and not penalize the next person (e.g., two golds mean the next gets silver, not bronze).
* Grouping rows into specific tiered ranking levels.

## Advantages
* Captures all valid data in a tier without arbitrarily cutting off records (unlike `ROW_NUMBER`).
* Maintains an intuitive consecutive tier system (1, 2, 3) rather than mathematical rank positioning (1, 3, 4).

## Limitations
* If the business logic requires strict top-N limits regardless of ties (e.g., "return exactly 3 rows"), `DENSE_RANK()` can return more than N rows if ties exist at the boundary.
* If gaps *are* expected (standard sports ranking where a tie for 1st means there is no 2nd place), `DENSE_RANK()` is incorrect.

## Common Comparisons
* **vs ROW_NUMBER():** `ROW_NUMBER` gives unique sequential IDs (1, 2, 3, 4). `DENSE_RANK` gives tied IDs (1, 1, 2, 3).
* **vs RANK():** `RANK` leaves gaps after ties (1, 1, 3, 4). `DENSE_RANK` never leaves gaps (1, 1, 2, 3).
* **vs LIMIT/OFFSET:** `LIMIT 1 OFFSET 1` can find a 2nd highest distinct value if preceded by a `SELECT DISTINCT`, but it's much harder to write for per-group (partitioned) queries.

## Common Interview Traps
* **Using `RANK()` for the Nth distinct value:** If two people tie for 1st, `RANK()` skips 2nd place entirely. Filtering for `rank = 2` will return zero rows!
* **Using `ROW_NUMBER()` for ties:** If two people tie for the 2nd highest salary, `ROW_NUMBER()` assigns one of them 2 and the other 3. You will lose data.
* **Forgetting PARTITION BY:** This causes the function to rank across the entire table instead of the required groups.

## SQL Syntax 
```sql
WITH RankedData AS (
  SELECT 
    column_name,
    DENSE_RANK() OVER (
      PARTITION BY grouping_column 
      ORDER BY value_column DESC
    ) AS dr
  FROM table_name
)
SELECT * FROM RankedData WHERE dr = N;
```

## 45-Second Interview Answer
"Whenever an interviewer asks for the 'Nth highest value', I immediately use DENSE_RANK(). Unlike ROW_NUMBER, which arbitrarily breaks ties, and RANK, which creates gaps after ties, DENSE_RANK assigns the same rank to matching values and ensures the very next value gets the next consecutive integer. I would wrap the DENSE_RANK function inside a CTE, partition by the necessary groups, order by the target metric, and then filter in the outer query for rank equals N."